# PestCLEF 2026 — Knowledge Graph Extraction Pipeline

**Task:** Document-level relation extraction → crop disease knowledge graph  
**Metric:** Macro-averaged F1 across documents  
**Pipeline:** BioBERT NER → Marker-based Relation Classification → KG → CSV



## 0. Install Dependencies

In [1]:
%%capture
!pip install -q seqeval transformers datasets accelerate scikit-learn tqdm rapidfuzz
print('✅ Dependencies installed')

## 1. Imports & Configuration

In [2]:
import os, re, json, time, random, itertools, csv, copy, datetime
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict, Counter
from typing import Optional, List, Dict, Tuple, Set

import numpy as np
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
from rapidfuzz import fuzz

from sklearn.metrics import f1_score as sk_f1
from seqeval.metrics import (
    classification_report as seq_report,
    f1_score, precision_score, recall_score,
)
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification, AutoConfig,
    DataCollatorForTokenClassification, TrainingArguments, Trainer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from datasets import Dataset

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected — switch to T4 GPU runtime for full training')

Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB


## 2. Mount Drive & Configure Paths

Set `BASE_DIR` to the folder in your Google Drive that contains:
- `train/`, `dev/`, `test/` — subdirectories with `.txt` document files
- `goldKG/train.json`, `goldKG/dev.json`, `goldKG/test.json` — annotation files


In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR       = Path('/content/drive/MyDrive/EPOP_documents')
TRAIN_TXT_DIR  = BASE_DIR / 'train'
DEV_TXT_DIR    = BASE_DIR / 'dev'
TEST_TXT_DIR   = BASE_DIR / 'test'
GOLD_DIR       = BASE_DIR / 'goldKG'
NER_MODEL_DIR  = BASE_DIR / 'models' / 'ner'
REL_MODEL_DIR  = BASE_DIR / 'models' / 'relation'
SUBMISSION_DIR = BASE_DIR / 'submissions'

for d in [NER_MODEL_DIR, REL_MODEL_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('✅ Paths configured')

Mounted at /content/drive
✅ Paths configured


## 3. Schema

Entity types, predicate labels, and valid (subject_type, object_type) pairs
taken directly from the EPOP corpus schema.


In [4]:
# ── Entity types & NER labels ─────────────────────────────────────────────────
ENTITY_TYPES = [
    'Pest', 'Plant', 'Disease', 'Vector',
    'Dissemination_pathway', 'Location', 'Date',
]
LABEL_LIST = ['O'] + [f'{b}-{e}' for e in ENTITY_TYPES for b in ('B', 'I')]
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}

# ── Relation predicates ────────────────────────────────────────────────────────
PREDICATES = [
    'NO_RELATION', 'Affects', 'Causes', 'Dispersed_by',
    'Found_on', 'Located_in', 'Occurs_on', 'Transmits',
]
PRED2ID  = {p: i for i, p in enumerate(PREDICATES)}
ID2PRED  = {i: p for p, i in PRED2ID.items()}
NO_REL_ID = PRED2ID['NO_RELATION']

# ── Valid (subject_type, object_type) → allowed predicates ────────────────────
VALID_PAIRS = {
    ('Disease', 'Plant'):                ['Affects'],
    ('Pest',    'Disease'):              ['Causes'],
    ('Disease', 'Dissemination_pathway'): ['Dispersed_by'],
    ('Pest',    'Dissemination_pathway'): ['Dispersed_by'],
    ('Pest',    'Plant'):                ['Found_on'],
    ('Vector',  'Plant'):                ['Found_on'],
    ('Vector',  'Dissemination_pathway'): ['Found_on'],
    ('Disease', 'Location'):             ['Located_in'],
    ('Pest',    'Location'):             ['Located_in'],
    ('Plant',   'Location'):             ['Located_in'],
    ('Vector',  'Location'):             ['Located_in'],
    ('Disease', 'Date'):                 ['Occurs_on'],
    ('Pest',    'Date'):                 ['Occurs_on'],
    ('Plant',   'Date'):                 ['Occurs_on'],
    ('Vector',  'Date'):                 ['Occurs_on'],
    ('Vector',  'Disease'):              ['Transmits'],
    ('Vector',  'Pest'):                 ['Transmits'],
}

# ── Role mapping for text_bound_annotations.relations ─────────────────────────
# Arg role names → subject vs. object slot
RELATION_ARG_SUBJECT_ROLES = {'Object', 'Organism', 'Nuisance'}
RELATION_ARG_OBJECT_ROLES  = {'Location', 'Habitat', 'Date', 'Dissemination_pathway'}


def predicate_normalize(p: str) -> str:
    """Case-fold and strip whitespace/underscores for edge matching."""
    return re.sub(r'[\s_]+', '', p.lower())


print(f'✅ Schema: {len(LABEL_LIST)} NER labels | '
      f'{len(PREDICATES)} predicates | {len(VALID_PAIRS)} valid entity-pair types')

✅ Schema: 15 NER labels | 8 predicates | 17 valid entity-pair types


## 4. Data Loading

`load_json_split()` is the single unified loader. It:
1. Reads the gold JSON file (`train.json` / `dev.json` / `test.json`)
2. Reads the matching `.txt` file for each `doc_id`
3. Parses `text_bound_annotations` entities with `start`/`end` char offsets
4. Resolves relation `args` role keys to subject/object entity spans
5. Parses `knowledge_graph` edges (arrays of accepted forms → first form used as canonical)


In [5]:
def _clean_text(text: str) -> str:
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return '\n'.join(l.strip() for l in text.split('\n') if len(l.strip()) > 2).strip()


def _parse_entities(tba: dict) -> dict:
    """
    Parse text_bound_annotations['entities'] into a flat dict keyed by entity ID.
    Each entity may have multiple offset spans; only the first is used.
    """
    eid_to_ent = {}
    for ent in tba.get('entities', []):
        offsets = ent.get('offsets', [])
        if not offsets:
            continue
        start, end = offsets[0]
        eid_to_ent[ent['id']] = {
            'id':           ent['id'],
            'type':         ent['type'],
            'text':         ent['form'],
            'start':        start,
            'end':          end,
            'canonical_id': (ent['normalizations'][0]['reference']
                             if ent.get('normalizations') else ''),
        }
    return eid_to_ent


def _parse_relations(tba: dict, eid_to_ent: dict) -> list:
    """
    Parse text_bound_annotations['relations'] using role-based arg keys.
    Roles in RELATION_ARG_SUBJECT_ROLES → subject slot.
    Roles in RELATION_ARG_OBJECT_ROLES  → object slot.
    """
    relations = []
    for rel in tba.get('relations', []):
        args     = rel.get('args', {})
        subj_eid = next((eid for role, eid in args.items()
                         if role in RELATION_ARG_SUBJECT_ROLES), None)
        obj_eid  = next((eid for role, eid in args.items()
                         if role in RELATION_ARG_OBJECT_ROLES), None)

        if not subj_eid or not obj_eid:
            continue
        if subj_eid not in eid_to_ent or obj_eid not in eid_to_ent:
            continue

        s = eid_to_ent[subj_eid]
        o = eid_to_ent[obj_eid]
        relations.append({
            'predicate':  rel['type'],
            'subject':    s['text'],  'subj_type':  s['type'],
            'subj_start': s['start'], 'subj_end':   s['end'],
            'object':     o['text'],  'obj_type':   o['type'],
            'obj_start':  o['start'], 'obj_end':    o['end'],
        })
    return relations


def _parse_kg_edges(kg_list: list) -> list:
    """Parse knowledge_graph edges; subject/object are arrays — first form is canonical."""
    edges = []
    for edge in kg_list:
        subj_forms = edge.get('subject', [])
        obj_forms  = edge.get('object',  [])
        if not subj_forms or not obj_forms:
            continue
        edges.append({
            'predicate':     edge['predicate'],
            'subject':       subj_forms[0],
            'subject_forms': subj_forms,
            'object':        obj_forms[0],
            'object_forms':  obj_forms,
        })
    return edges


def load_json_split(json_path, txt_dir, verbose=True) -> list:
    """
    Unified loader for train / dev / test splits.

    Returns a list of doc dicts:
        {
          'id':        str,
          'raw_text':  str,          # from .txt file
          'entities':  list[dict],   # from text_bound_annotations (empty for test)
          'relations': list[dict],   # from text_bound_annotations (empty for test)
          'kg_edges':  list[dict],   # from knowledge_graph        (empty for test)
        }
    """
    json_path = Path(json_path)
    txt_dir   = Path(txt_dir)

    if not json_path.exists():
        print(f'[WARN] JSON not found: {json_path}')
        return []

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    docs, missing_txt = [], 0

    for item in data:
        doc_id   = item.get('doc_id', 'unknown')
        txt_path = txt_dir / f'{doc_id}.txt'

        if txt_path.exists():
            raw_text = _clean_text(txt_path.read_text('utf-8', errors='replace'))
        else:
            raw_text = ''
            missing_txt += 1

        tba        = item.get('text_bound_annotations', {})
        eid_to_ent = _parse_entities(tba)
        entities   = list(eid_to_ent.values())
        relations  = _parse_relations(tba, eid_to_ent)
        kg_edges   = _parse_kg_edges(item.get('knowledge_graph', []))

        docs.append({
            'id':        doc_id,
            'raw_text':  raw_text,
            'entities':  entities,
            'relations': relations,
            'kg_edges':  kg_edges,
        })

    if verbose:
        has_gold = sum(1 for d in docs if d['entities'])
        avg_ents = np.mean([len(d['entities'])  for d in docs]) if docs else 0
        avg_rels = np.mean([len(d['relations']) for d in docs]) if docs else 0
        avg_kg   = np.mean([len(d['kg_edges'])  for d in docs]) if docs else 0
        print(f'[{json_path.stem}] {len(docs)} docs | {has_gold} with gold annotations')
        print(f'  avg entities={avg_ents:.1f} | avg relations={avg_rels:.1f} | '
              f'avg KG edges={avg_kg:.1f}')
        if missing_txt:
            print(f'  ⚠️  {missing_txt} docs missing .txt file')

    return docs

print('✅ Data loading functions ready')

✅ Data loading functions ready


## 5. Load Splits

In [6]:
train_docs = load_json_split(GOLD_DIR / 'train.json', TRAIN_TXT_DIR)
dev_docs   = load_json_split(GOLD_DIR / 'dev.json',   DEV_TXT_DIR)
test_docs  = load_json_split(GOLD_DIR / 'test.json',  TEST_TXT_DIR)

print(f'\nLoaded: {len(train_docs)} train | {len(dev_docs)} dev | {len(test_docs)} test')

[train] 110 docs | 110 with gold annotations
  avg entities=31.7 | avg relations=17.9 | avg KG edges=12.5
[dev] 55 docs | 54 with gold annotations
  avg entities=29.4 | avg relations=16.4 | avg KG edges=11.9
[test] 82 docs | 0 with gold annotations
  avg entities=0.0 | avg relations=0.0 | avg KG edges=0.0

Loaded: 110 train | 55 dev | 82 test


## 6. Silver Labeling (Test-Set Fallback)

Rule-based entity extraction used **only** when a document has no gold
`text_bound_annotations` (i.e. the test set). Train and dev always use gold annotations.


In [7]:
# ── Term dictionaries ─────────────────────────────────────────────────────────
PEST_TERMS = [
    'Phytophthora infestans', 'Plasmopara viticola', 'Botrytis cinerea',
    'Fusarium oxysporum', 'Puccinia striiformis', 'Alternaria solani',
    'Blumeria graminis', 'Verticillium dahliae', 'Xylella fastidiosa',
    'Ralstonia solanacearum', 'Erwinia amylovora', 'Bursaphelenchus xylophilus',
    'Meloidogyne incognita', 'Globodera pallida', 'Myzus persicae',
    'Bemisia tabaci', 'Trialeurodes vaporariorum', 'Frankliniella occidentalis',
    'Spodoptera frugiperda', 'Helicoverpa armigera', 'Tuta absoluta',
    'Drosophila suzukii', 'Bactrocera dorsalis', 'Ceratitis capitata',
    'fall armyworm', 'tomato leafminer', 'spotted wing drosophila',
    'green peach aphid', 'silverleaf whitefly', 'western flower thrips',
    'pine wood nematode', 'medfly',
    'ToBRFV', 'TYLCV', 'TSWV', 'PVY', 'PPV', 'HLB',
]
PLANT_TERMS = [
    'tomato', 'potato', 'wheat', 'rice', 'maize', 'corn', 'soybean', 'grape',
    'grapevine', 'apple', 'citrus', 'orange', 'lemon', 'cotton', 'barley',
    'sunflower', 'pepper', 'cucumber', 'strawberry', 'olive', 'peach', 'plum',
    'banana', 'coffee', 'sugar beet', 'pine',
    'Solanum tuberosum', 'Solanum lycopersicum', 'Triticum aestivum',
    'Oryza sativa', 'Zea mays', 'Vitis vinifera', 'Malus domestica',
]
DISEASE_TERMS = [
    'late blight', 'early blight', 'powdery mildew', 'downy mildew',
    'gray mold', 'grey mold', 'fire blight', 'citrus greening', 'bacterial wilt',
    'huanglongbing', 'yellow rust', 'stripe rust',
]
VECTOR_TERMS = [
    'aphid', 'whitefly', 'thrips', 'leafhopper', 'psyllid', 'mealybug',
    'Myzus persicae', 'Bemisia tabaci', 'Frankliniella occidentalis',
]
PATHWAY_TERMS = [
    'infected plant material', 'contaminated soil', 'wood packaging',
    'wooden crates', 'seed', 'seeds',
]
LOCATION_TERMS = [
    'France', 'Spain', 'Italy', 'Germany', 'USA', 'United States', 'China',
    'India', 'Brazil', 'Australia', 'Europe', 'Africa', 'Asia', 'Mediterranean',
    'North America', 'South America', 'Jordan', 'Jordan Valley',
]

_BAD_TERMS = {'a', 'an', 'the', 'of', 'to', 'in', 'on', 'for', 'by', 'with', 'and'}
TERM_DICT  = {}
for t in PEST_TERMS:     TERM_DICT[t] = 'Pest'
for t in PLANT_TERMS:    TERM_DICT[t] = 'Plant'
for t in DISEASE_TERMS:  TERM_DICT[t] = 'Disease'
for t in VECTOR_TERMS:   TERM_DICT[t] = 'Vector'
for t in PATHWAY_TERMS:  TERM_DICT[t] = 'Dissemination_pathway'
for t in LOCATION_TERMS: TERM_DICT[t] = 'Location'
TERM_DICT = {k: v for k, v in TERM_DICT.items()
             if len(k) > 3 and k not in _BAD_TERMS}

PREDICATE_MAP = {
    ('Pest',    'Disease'):               'Causes',
    ('Disease', 'Plant'):                 'Affects',
    ('Vector',  'Pest'):                  'Transmits',
    ('Vector',  'Disease'):               'Transmits',
    ('Pest',    'Dissemination_pathway'): 'Dispersed_by',
    ('Disease', 'Dissemination_pathway'): 'Dispersed_by',
    ('Pest',    'Plant'):                 'Found_on',
    ('Vector',  'Plant'):                 'Found_on',
    ('Pest',    'Location'):              'Located_in',
    ('Disease', 'Location'):              'Located_in',
    ('Plant',   'Location'):              'Located_in',
    ('Vector',  'Location'):              'Located_in',
    ('Pest',    'Date'):                  'Occurs_on',
    ('Disease', 'Date'):                  'Occurs_on',
    ('Plant',   'Date'):                  'Occurs_on',
    ('Vector',  'Date'):                  'Occurs_on',
}


def _extract_entities_silver(text: str) -> list:
    entities, used_spans = [], []
    for term, etype in sorted(TERM_DICT.items(), key=lambda x: -len(x[0])):
        for m in re.finditer(r'\b' + re.escape(term) + r'\b', text, re.IGNORECASE):
            s, e    = m.start(), m.end()
            span_text = text[s:e].strip()
            if len(span_text) < 3 or not any(c.isalpha() for c in span_text):
                continue
            if any(not (e <= ss or s >= se) for ss, se in used_spans):
                continue
            entities.append({'text': span_text, 'type': etype,
                              'start': s, 'end': e, 'score': 1.0, 'canonical_id': term})
            used_spans.append((s, e))
    return entities


def silver_label_doc(doc: dict) -> dict:
    """Fallback silver labeling — skipped if doc already has gold entities."""
    if doc.get('entities'):
        return doc
    text     = doc['raw_text']
    entities = _extract_entities_silver(text)
    relations, seen = [], set()
    for sent in re.split(r'(?<=[.!?])\s+', text):
        ss      = text.find(sent)
        se      = ss + len(sent)
        present = [e for e in entities if e['start'] >= ss and e['end'] <= se]
        for e1, e2 in itertools.permutations(present, 2):
            pair = (e1['type'], e2['type'])
            if pair not in PREDICATE_MAP:
                continue
            key = (e1['text'], PREDICATE_MAP[pair], e2['text'])
            if key in seen:
                continue
            seen.add(key)
            relations.append({
                'subject': e1['text'], 'subj_type': e1['type'],
                'subj_start': e1['start'], 'subj_end': e1['end'],
                'object': e2['text'], 'obj_type': e2['type'],
                'obj_start': e2['start'], 'obj_end': e2['end'],
                'predicate': PREDICATE_MAP[pair], 'score': 0.7,
            })
    doc['entities']  = entities
    doc['relations'] = relations
    return doc

print('✅ Silver labeling ready (fallback only)')

✅ Silver labeling ready (fallback only)


## 7. Apply Silver Labels to Test Set

In [8]:
print('Applying silver labeling to test set (gold not available)...')
for doc in tqdm(test_docs, desc='Test silver'):
    silver_label_doc(doc)

# Sanity check
for split_name, docs in [('train', train_docs), ('dev', dev_docs), ('test', test_docs)]:
    with_ents  = sum(1 for d in docs if d['entities'])
    total_ents = sum(len(d['entities'])  for d in docs)
    total_rels = sum(len(d['relations']) for d in docs)
    print(f'{split_name:5s}: {with_ents}/{len(docs)} docs have entities | '
          f'{total_ents} entities | {total_rels} relations')

Applying silver labeling to test set (gold not available)...


Test silver:   0%|          | 0/82 [00:00<?, ?it/s]

train: 110/110 docs have entities | 3482 entities | 1974 relations
dev  : 54/55 docs have entities | 1615 entities | 900 relations
test : 79/82 docs have entities | 823 entities | 197 relations


## 8. NER — BIO Conversion & BioBERT Model

`doc_to_hf_from_gold()` builds BIO labels from gold entity char offsets.
The `PestNERModel` wraps a BioBERT token classifier with:
- Weighted cross-entropy loss (compensates for heavy O-label imbalance)
- Sliding-window inference for long documents


In [9]:
BASE_NER_MODEL = 'dmis-lab/biobert-base-cased-v1.2'


def doc_to_hf_from_gold(doc: dict, tokenizer) -> Optional[dict]:
    """
    Convert a doc with gold entity offsets to an HF-format NER example.

    1. Split raw_text on whitespace → word spans
    2. For each gold entity: find overlapping words → assign B-/I- labels
    3. Return {'tokens': [...], 'ner_tags': [...]} (word-level int labels)
    """
    text     = doc['raw_text']
    entities = doc.get('entities', [])
    if not text.strip():
        return None

    word_spans  = [(m.start(), m.end(), m.group()) for m in re.finditer(r'\S+', text)]
    if not word_spans:
        return None

    words       = [w for _, _, w in word_spans]
    word_labels = ['O'] * len(word_spans)

    for ent in entities:
        et, es, ee = ent.get('type'), ent.get('start'), ent.get('end')
        if et not in ENTITY_TYPES or es is None or ee is None:
            continue
        if not (0 <= es < ee <= len(text)):
            continue
        started = False
        for i, (ws, we, _) in enumerate(word_spans):
            if not (we <= es or ws >= ee):        # overlap
                word_labels[i] = f'B-{et}' if not started else f'I-{et}'
                started = True

    if all(l == 'O' for l in word_labels):        # skip all-O examples
        return None

    return {
        'tokens':   words,
        'ner_tags': [LABEL2ID.get(lbl, 0) for lbl in word_labels],
    }


def verify_ner_labels(docs: list, tokenizer, n_sample: int = 5, label: str = 'split'):
    """Quick sanity check: report entity token density."""
    total_tokens, nonO_tokens = 0, 0
    for doc in docs:
        hf = doc_to_hf_from_gold(doc, tokenizer)
        if not hf:
            continue
        total_tokens += len(hf['ner_tags'])
        nonO_tokens  += sum(1 for t in hf['ner_tags'] if t != 0)
    pct = 100 * nonO_tokens / max(1, total_tokens)
    print(f'[{label}] {total_tokens} tokens | {nonO_tokens} non-O ({pct:.2f}%)')
    if pct < 0.1:
        print('  ❌ CRITICAL: Almost no entity labels — check gold annotations!')
    elif pct < 1.0:
        print('  ⚠️  Low entity density')
    else:
        print('  ✅ Entity density OK')


def build_ner_class_weights(train_hf: list, label_list: list,
                             max_weight: float = 5.0) -> torch.Tensor:
    """Inverse-frequency class weights for weighted NER loss."""
    counts = Counter(t for ex in train_hf for t in ex['ner_tags'] if t != -100)
    total  = sum(counts.values())
    weights = []
    for i, label in enumerate(label_list):
        freq = counts.get(i, 0)
        if label == 'O' or freq == 0:
            weights.append(1.0)
        else:
            weights.append(min(total / (len(label_list) * freq), max_weight))
    return torch.tensor(weights, dtype=torch.float)


class WeightedTrainer(Trainer):
    """HF Trainer with per-class weighted cross-entropy loss."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get('labels')
        outputs = model(**inputs)
        logits  = outputs.get('logits')
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
            if self.class_weights is not None else None,
            ignore_index=-100,
        )
        loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class PestNERModel:
    """BioBERT-based NER model with sliding-window inference."""

    def __init__(self, base_model: str = BASE_NER_MODEL, max_length: int = 512):
        self.max_length = max_length
        self.tokenizer  = AutoTokenizer.from_pretrained(base_model)
        self.model      = AutoModelForTokenClassification.from_pretrained(
            base_model, num_labels=len(LABEL_LIST),
            id2label=ID2LABEL, label2id=LABEL2ID,
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
        print(f'[NER] {base_model} | {len(LABEL_LIST)} labels | {DEVICE}')

    @classmethod
    def from_pretrained(cls, model_dir):
        obj = cls.__new__(cls)
        obj.max_length = 512
        obj.tokenizer  = AutoTokenizer.from_pretrained(str(model_dir))
        obj.model      = AutoModelForTokenClassification.from_pretrained(
            str(model_dir)).to(DEVICE)
        print(f'[NER] Loaded from {model_dir}')
        return obj

    def _tokenize_and_align(self, examples):
        tok = self.tokenizer(
            examples['tokens'],
            truncation=True, max_length=self.max_length,
            is_split_into_words=True, padding=False,
        )
        all_labels = []
        for i, word_labels in enumerate(examples['ner_tags']):
            word_ids = tok.word_ids(batch_index=i)
            prev_wid, aligned = None, []
            for wid in word_ids:
                if wid is None:
                    aligned.append(-100)
                elif wid != prev_wid:
                    aligned.append(word_labels[wid])
                else:
                    aligned.append(-100)
                prev_wid = wid
            all_labels.append(aligned)
        tok['labels'] = all_labels
        return tok

    def _compute_metrics(self, p):
        predictions, labels = p
        predictions = predictions.argmax(axis=2)
        true_preds, true_labels = [], []
        for pred, lab in zip(predictions, labels):
            cp, cl = [], []
            for pi, li in zip(pred, lab):
                if li != -100:
                    cp.append(ID2LABEL[pi])
                    cl.append(ID2LABEL[li])
            true_preds.append(cp); true_labels.append(cl)
        return {
            'eval_precision': precision_score(true_labels, true_preds),
            'eval_recall':    recall_score(true_labels, true_preds),
            'eval_f1':        f1_score(true_labels, true_preds),
        }

    def train(self, train_docs: list, eval_docs: list, output_dir,
              epochs: int = 8, batch_size: int = 16, lr: float = 3e-5):
        train_hf = [x for x in
                    (doc_to_hf_from_gold(d, self.tokenizer)
                     for d in train_docs if d.get('entities')) if x]
        eval_hf  = [x for x in
                    (doc_to_hf_from_gold(d, self.tokenizer)
                     for d in eval_docs  if d.get('entities')) if x]

        if not train_hf:
            print('[NER] ❌ No training examples — check gold annotations')
            return

        all_tags = [t for ex in train_hf for t in ex['ner_tags']]
        nonO     = sum(1 for t in all_tags if t != 0)
        pct      = 100 * nonO / max(1, len(all_tags))
        print(f'[NER] {len(train_hf)} train docs | {len(all_tags)} tokens | '
              f'{nonO} entity tokens ({pct:.2f}%)')
        if pct < 0.05:
            print('[NER] ❌ ABORTING — entity label rate too low')
            return

        train_ds = Dataset.from_list(train_hf).map(self._tokenize_and_align, batched=True)
        eval_ds  = Dataset.from_list(eval_hf).map(self._tokenize_and_align,  batched=True)

        total_steps  = (len(train_ds) // batch_size) * epochs
        warmup_steps = int(0.1 * total_steps)
        args = TrainingArguments(
            output_dir=str(output_dir),
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            weight_decay=0.01,
            warmup_steps=warmup_steps,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_f1',
            fp16=(DEVICE == 'cuda'),
            report_to='none',
            logging_steps=20,
        )
        class_weights = build_ner_class_weights(train_hf, LABEL_LIST, max_weight=5.0)
        trainer = WeightedTrainer(
            model=self.model, args=args,
            train_dataset=train_ds, eval_dataset=eval_ds,
            processing_class=self.tokenizer,
            data_collator=DataCollatorForTokenClassification(self.tokenizer),
            compute_metrics=self._compute_metrics,
            class_weights=class_weights,
        )
        trainer.train()
        trainer.save_model(str(output_dir))
        self.tokenizer.save_pretrained(str(output_dir))
        print(f'[NER] ✅ Saved → {output_dir}')

    def predict(self, text: str) -> list:
        """Sliding-window inference; returns list of entity dicts."""
        self.model.eval()
        enc     = self.tokenizer(text, return_offsets_mapping=True,
                                 add_special_tokens=True, truncation=False)
        ids     = enc['input_ids']
        offsets = enc['offset_mapping']
        stride  = self.max_length - 50
        all_lid = [0]   * len(ids)
        all_sc  = [0.0] * len(ids)
        for start in range(0, len(ids), stride):
            end = min(start + self.max_length, len(ids))
            inp = torch.tensor([ids[start:end]]).to(DEVICE)
            with torch.no_grad():
                logits = self.model(inp).logits
            probs = torch.softmax(logits, dim=-1)[0]
            preds = torch.argmax(probs, dim=-1)
            for i in range(end - start):
                if start + i < len(all_lid):
                    all_lid[start + i] = preds[i].item()
                    all_sc[start + i]  = probs[i, preds[i]].item()
        return self._decode(text, offsets, all_lid, all_sc)

    def _decode(self, text: str, offsets, lids, scores) -> list:
        ents = []
        ct = cs = ce = None; csc = 0.0
        for (s, e), lid, sc in zip(offsets, lids, scores):
            lbl = ID2LABEL[lid]
            if lbl.startswith('B-'):
                if ct:
                    sp = text[cs:ce].strip()
                    if sp and csc >= 0.5 and len(sp) >= 3:
                        ents.append({'text': sp, 'type': ct, 'start': cs,
                                     'end': ce, 'score': csc, 'canonical_id': ''})
                ct, cs, ce, csc = lbl[2:], s, e, sc
            elif lbl.startswith('I-') and lbl[2:] == ct:
                ce = e; csc = min(csc, sc)
            else:
                if ct:
                    sp = text[cs:ce].strip()
                    if sp and csc >= 0.5 and len(sp) >= 3:
                        ents.append({'text': sp, 'type': ct, 'start': cs,
                                     'end': ce, 'score': csc, 'canonical_id': ''})
                ct = cs = ce = None; csc = 0.0
        if ct:
            sp = text[cs:ce].strip()
            if sp and csc >= 0.5 and len(sp) >= 3:
                ents.append({'text': sp, 'type': ct, 'start': cs,
                             'end': ce, 'score': csc, 'canonical_id': ''})
        return ents

print('✅ NER model class ready')

✅ NER model class ready


## 9. Relation Classification Model

Marker-based BioBERT classifier:
- Entity spans are delimited with `[SUBJ_START]`/`[SUBJ_END]` and `[OBJ_START]`/`[OBJ_END]` tokens
- Subject & object marker representations are concatenated with distance features
- Weighted cross-entropy handles class imbalance across predicates


In [10]:
SUBJ_START, SUBJ_END   = '[SUBJ_START]', '[SUBJ_END]'
OBJ_START,  OBJ_END    = '[OBJ_START]',  '[OBJ_END]'
SPECIAL_TOKENS         = [SUBJ_START, SUBJ_END, OBJ_START, OBJ_END]


class RelationClassifier(nn.Module):
    """BioBERT encoder + marker-pooled classifier head."""

    def __init__(self, encoder_name: str = BASE_NER_MODEL,
                 num_labels: int = len(PREDICATES), dropout: float = 0.15):
        super().__init__()
        cfg          = AutoConfig.from_pretrained(encoder_name)
        self.encoder = AutoModel.from_pretrained(encoder_name)
        H            = cfg.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H * 2 + 3, H),
            nn.GELU(),
            nn.LayerNorm(H),
            nn.Dropout(dropout),
            nn.Linear(H, num_labels),
        )

    def forward(self, input_ids, attention_mask,
                subj_pos, obj_pos, distance, subj_norm, obj_norm):
        h = self.encoder(input_ids=input_ids,
                         attention_mask=attention_mask).last_hidden_state
        B = h.size(0)
        pooled = torch.cat([
            h[torch.arange(B), subj_pos],
            h[torch.arange(B), obj_pos],
            distance.unsqueeze(1),
            subj_norm.unsqueeze(1),
            obj_norm.unsqueeze(1),
        ], dim=-1)
        return self.classifier(pooled)


@dataclass
class RelExample:
    doc_id:     str;  text:       str
    subj_text:  str;  subj_type:  str;  subj_start: int;  subj_end: int
    obj_text:   str;  obj_type:   str;  obj_start:  int;  obj_end:  int
    label:      int = 0


def insert_markers(text: str, ss: int, se: int, os_: int, oe: int) -> str:
    """Insert entity markers into text at the given char offsets."""
    spans = sorted(
        [(ss, SUBJ_START), (se, SUBJ_END), (os_, OBJ_START), (oe, OBJ_END)],
        key=lambda x: x[0], reverse=True,
    )
    for pos, marker in spans:
        pos  = max(0, min(pos, len(text)))
        text = text[:pos] + f' {marker} ' + text[pos:]
    return text


def build_relation_examples_from_gold(docs: list, neg_ratio: float = 3.0) -> list:
    """
    Build RelExample list from gold relations (train/dev).

    Positives  : from doc['relations'] (resolved from text_bound_annotations)
    Hard negatives: valid-pair entity permutations not already in positives,
                    within 300 chars, sampled at neg_ratio × positives per doc.
    """
    examples = []
    for doc in docs:
        text, ents, rels = doc['raw_text'], doc.get('entities', []), doc.get('relations', [])
        pos_set, positives = set(), []

        for r in rels:
            pred = r.get('predicate', '')
            if pred not in PRED2ID or PRED2ID[pred] == NO_REL_ID:
                continue
            s = {'text': r['subject'],  'type': r['subj_type'],
                 'start': r['subj_start'], 'end': r['subj_end']}
            o = {'text': r['object'],   'type': r['obj_type'],
                 'start': r['obj_start'],  'end': r['obj_end']}
            examples.append(RelExample(
                doc['id'], text,
                s['text'], s['type'], s['start'], s['end'],
                o['text'], o['type'], o['start'], o['end'],
                label=PRED2ID[pred],
            ))
            positives.append((s, o))
            pos_set.add((s['text'], o['text']))

        hard_negs = [
            (s, o)
            for s, o in itertools.permutations(ents, 2)
            if (s['text'], o['text']) not in pos_set
            and (s['type'], o['type']) in VALID_PAIRS
            and abs(s.get('start', 0) - o.get('start', 0)) <= 300
            and s['text'].lower() != o['text'].lower()
        ]
        random.shuffle(hard_negs)
        for s, o in hard_negs[:int(len(positives) * neg_ratio)]:
            examples.append(RelExample(
                doc['id'], text,
                s['text'], s['type'], s.get('start', 0), s.get('end', 0),
                o['text'], o['type'], o.get('start', 0), o.get('end', 0),
                label=NO_REL_ID,
            ))

    pos = sum(1 for e in examples if e.label != NO_REL_ID)
    print(f'✅ {len(examples)} RE examples: {pos} positive | {len(examples) - pos} negative')
    return examples


class RelationModel:
    """Marker-based BioBERT relation classifier."""

    def __init__(self, base_model: str = BASE_NER_MODEL, max_length: int = 512):
        self.max_length = max_length
        self.tokenizer  = AutoTokenizer.from_pretrained(base_model)
        self.tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)
        self.model      = RelationClassifier(encoder_name=base_model).to(DEVICE)
        self.model.encoder.resize_token_embeddings(len(self.tokenizer))
        print(f'[REL] {base_model} | {len(PREDICATES)} predicates | {DEVICE}')

    @classmethod
    def from_pretrained(cls, d):
        obj = cls.__new__(cls); obj.max_length = 512
        obj.tokenizer = AutoTokenizer.from_pretrained(str(d))
        obj.model     = torch.load(Path(d) / 'relation_model.pt', map_location=DEVICE)
        obj.model.eval()
        print(f'[REL] Loaded from {d}')
        return obj

    def save(self, d):
        Path(d).mkdir(parents=True, exist_ok=True)
        torch.save(self.model, Path(d) / 'relation_model.pt')
        self.tokenizer.save_pretrained(str(d))
        print(f'[REL] ✅ Saved → {d}')

    def _encode(self, ex: RelExample) -> dict:
        marked = insert_markers(ex.text,
                                ex.subj_start, ex.subj_end,
                                ex.obj_start,  ex.obj_end)
        enc  = self.tokenizer(marked, max_length=self.max_length,
                              truncation=True, padding='max_length',
                              return_tensors='pt')
        ids  = enc['input_ids'][0]
        sid  = self.tokenizer.convert_tokens_to_ids(SUBJ_START)
        oid  = self.tokenizer.convert_tokens_to_ids(OBJ_START)
        sp   = (ids == sid).nonzero(as_tuple=True)[0]
        op   = (ids == oid).nonzero(as_tuple=True)[0]
        subj_pos  = sp[0].item() if len(sp) else 1
        obj_pos   = op[0].item() if len(op) else 2
        subj_norm = subj_pos / self.max_length
        obj_norm  = obj_pos  / self.max_length
        distance  = min(abs(subj_pos - obj_pos), 512) / 512.0
        return {
            'input_ids':       ids,
            'attention_mask':  enc['attention_mask'][0],
            'subj_marker_pos': torch.tensor(subj_pos),
            'obj_marker_pos':  torch.tensor(obj_pos),
            'distance':        torch.tensor(distance, dtype=torch.float),
            'subj_norm':       torch.tensor(subj_norm, dtype=torch.float),
            'obj_norm':        torch.tensor(obj_norm,  dtype=torch.float),
            'labels':          torch.tensor(ex.label),
        }

    def _batch_forward(self, batch: list):
        return self.model(
            torch.stack([x['input_ids']       for x in batch]).to(DEVICE),
            torch.stack([x['attention_mask']  for x in batch]).to(DEVICE),
            torch.stack([x['subj_marker_pos'] for x in batch]).to(DEVICE),
            torch.stack([x['obj_marker_pos']  for x in batch]).to(DEVICE),
            torch.stack([x['distance']        for x in batch]).to(DEVICE),
            torch.stack([x['subj_norm']       for x in batch]).to(DEVICE),
            torch.stack([x['obj_norm']        for x in batch]).to(DEVICE),
        )

    def train(self, train_ex: list, eval_ex: list, output_dir,
              epochs: int = 5, batch_size: int = 16, lr: float = 2e-5):
        counts       = torch.bincount(torch.tensor([e.label for e in train_ex]),
                                      minlength=len(PREDICATES)).float()
        total        = counts.sum()
        class_weights = torch.clamp(torch.log1p(total / (counts + 1)),
                                    min=0.5, max=10.0).to(DEVICE)
        print(f'[REL] Class weights: {[round(w, 3) for w in class_weights.tolist()]}')
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        opt     = AdamW(self.model.parameters(), lr=lr, weight_decay=0.01)
        steps   = max(1, len(train_ex) // batch_size) * epochs
        sched   = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)

        tr_enc = [self._encode(e) for e in tqdm(train_ex, desc='Encode train')]
        ev_enc = [self._encode(e) for e in tqdm(eval_ex,  desc='Encode eval')]
        best   = 0.0

        for ep in range(epochs):
            self.model.train()
            total_loss, n = 0, 0
            idx = torch.randperm(len(tr_enc))
            for start in tqdm(range(0, len(tr_enc), batch_size),
                              desc=f'Epoch {ep + 1}/{epochs}', leave=False):
                b      = [tr_enc[i] for i in idx[start:start + batch_size]]
                opt.zero_grad()
                logits = self._batch_forward(b)
                labels = torch.stack([x['labels'] for x in b]).to(DEVICE)
                loss   = loss_fn(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step(); sched.step()
                total_loss += loss.item(); n += 1
            f1 = self._eval(ev_enc, batch_size)
            print(f'  Epoch {ep + 1}/{epochs}  loss={total_loss / max(1, n):.4f}  F1={f1:.4f}')
            if f1 > best:
                best = f1; self.save(output_dir)
        print(f'[REL] Best F1: {best:.4f}')

    def _eval(self, enc: list, bs: int = 16) -> float:
        self.model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for start in range(0, len(enc), bs):
                b      = enc[start:start + bs]
                logits = self._batch_forward(b)
                preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
                labels.extend(x['labels'].item() for x in b)
        pos = [i for i in range(len(PREDICATES)) if i != NO_REL_ID]
        return sk_f1(labels, preds, labels=pos, average='macro', zero_division=0)

    def predict_document(self, text: str, entities: list,
                          threshold: float = 0.45) -> list:
        """Predict KG edges for all valid entity pairs in a document."""
        self.model.eval()
        rels = []
        for s, o in itertools.permutations(entities, 2):
            if (s['type'], o['type']) not in VALID_PAIRS:
                continue
            ex  = RelExample('inf', text,
                             s['text'], s['type'], s.get('start', 0), s.get('end', 0),
                             o['text'], o['type'], o.get('start', 0), o.get('end', 0))
            enc = self._encode(ex)
            with torch.no_grad():
                logits = self._batch_forward([enc])
                probs  = torch.softmax(logits, dim=-1)[0]
                pid    = torch.argmax(probs).item()
                score  = probs[pid].item()
            if ID2PRED[pid] != 'NO_RELATION' and score >= threshold:
                rels.append({
                    'subject':   s.get('canonical_id') or s['text'],
                    'predicate': ID2PRED[pid],
                    'object':    o.get('canonical_id') or o['text'],
                    'score':     score,
                })
        return rels

print('✅ Relation model class ready')

✅ Relation model class ready


## 10. KG Assembly & Evaluation

`build_kg()` deduplicates edges using canonical entity IDs.  
`doc_f1()` implements the competition's exact edge-matching metric.  
`format_submission()` writes the required CSV format.


In [11]:
def norm_edge(s: str, p: str, o: str) -> tuple:
    """Normalised edge key for dedup and matching (case-fold, strip whitespace/underscores)."""
    return (s.strip(), predicate_normalize(p), o.strip())


def build_kg(entities: list, relations: list,
             score_threshold: float = 0.35) -> list:
    """Assemble deduplicated KG edges from predicted relations."""
    em = {}
    for e in entities:
        cid = e.get('canonical_id') or e['text']
        em[e['text']] = cid; em[cid] = cid

    edges, seen = [], set()
    for r in sorted(relations, key=lambda x: -x.get('score', 0)):
        s, o = r.get('subject', ''), r.get('object', '')
        if not s or not o or s.lower() == o.lower():
            continue
        if r.get('score', 1) < score_threshold:
            continue
        s_c = em.get(s, s)
        o_c = em.get(o, o)
        k   = norm_edge(s_c, r['predicate'], o_c)
        if k not in seen:
            seen.add(k)
            edges.append({'subject': s_c, 'predicate': r['predicate'], 'object': o_c})
    return edges


def doc_f1(pred: list, ref: list) -> float:
    """Competition F1 for a single document."""
    if not ref and not pred: return 1.0
    if not ref or  not pred: return 0.0
    def to_counter(edges):
        return Counter(norm_edge(e['subject'], e['predicate'], e['object'])
                       for e in edges)
    pc, rc = to_counter(pred), to_counter(ref)
    tp = sum((pc & rc).values())
    P  = tp / len(pred)
    R  = tp / len(ref)
    return 2 * P * R / (P + R) if (P + R) > 0 else 0.0


def evaluate(predictions: dict, references: dict, verbose: bool = False) -> dict:
    """
    Compute macro-averaged F1 across all documents.

    predictions : doc_id → list of edge dicts
    references  : doc_id → list of edge dicts (from kg_edges)
    """
    per_doc = {}
    for doc_id, ref in references.items():
        pred          = predictions.get(doc_id, [])
        f1            = doc_f1(pred, ref)
        per_doc[doc_id] = f1
        if verbose:
            icon = '✓' if f1 >= 0.5 else ('~' if f1 > 0 else '✗')
            print(f'  {icon} {doc_id:<35} F1={f1:.4f}  pred={len(pred):<4} ref={len(ref)}')
    macro = sum(per_doc.values()) / len(per_doc) if per_doc else 0.0
    print(f'\n  MACRO F1 = {macro:.4f}  ({len(per_doc)} docs)')
    return {'macro_f1': round(macro, 6), 'per_doc': per_doc}


def format_submission(submission: dict, output_path) -> None:
    """Write submission.csv in the required competition format."""
    rows = []
    for doc_id, edges in submission.items():
        kg = json.dumps(
            [{'predicate': e['predicate'], 'subject': e['subject'], 'object': e['object']}
             for e in edges],
            ensure_ascii=False,
        )
        rows.append({'doc_id': doc_id, 'knowledge_graph': kg})
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['doc_id', 'knowledge_graph'],
                           quoting=csv.QUOTE_ALL)
        w.writeheader(); w.writerows(rows)
    total = sum(len(v) for v in submission.values())
    print(f'✅ Submission: {len(submission)} docs | {total} edges → {output_path}')

print('✅ KG assembly and evaluation ready')

✅ KG assembly and evaluation ready


## 11. Diagnose NER Labels (Run Before Training)

In [12]:
_diag_tok = AutoTokenizer.from_pretrained(BASE_NER_MODEL)
print('=== Train set NER label diagnosis ===')
verify_ner_labels(train_docs, _diag_tok, label='train')
print('\n=== Dev set NER label diagnosis ===')
verify_ner_labels(dev_docs, _diag_tok, label='dev')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

=== Train set NER label diagnosis ===
[train] 52947 tokens | 6568 non-O (12.40%)
  ✅ Entity density OK

=== Dev set NER label diagnosis ===
[dev] 25120 tokens | 3002 non-O (11.95%)
  ✅ Entity density OK


## 12. Train NER Model

In [13]:
ner_model = PestNERModel()
ner_model.train(
    train_docs=train_docs,
    eval_docs=dev_docs,
    output_dir=NER_MODEL_DIR,
)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

[NER] dmis-lab/biobert-base-cased-v1.2 | 15 labels | cuda
[NER] 110 train docs | 52947 tokens | 6568 entity tokens (12.40%)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[NER] ✅ Saved → /content/drive/MyDrive/EPOP_documents/models/ner


## 13. Train Relation Model

In [14]:
# Use train + dev for final model (no dev leakage risk since test has no gold labels)
examples = build_relation_examples_from_gold(train_docs + dev_docs, neg_ratio=3.0)
random.shuffle(examples)
val_n     = max(1, int(len(examples) * 0.15))
rel_model = RelationModel()
rel_model.train(
    train_ex=examples[val_n:],
    eval_ex=examples[:val_n],
    output_dir=REL_MODEL_DIR,
    epochs=5,
)

✅ 7758 RE examples: 2874 positive | 4884 negative


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As describe

[REL] dmis-lab/biobert-base-cased-v1.2 | 8 predicates | cuda
[REL] Class weights: [0.946, 8.794, 8.794, 4.994, 2.616, 1.689, 2.972, 8.794]


Encode train:   0%|          | 0/6595 [00:00<?, ?it/s]

...

  Epoch 3/5  loss=0.5259  F1=0.3719


Epoch 4/5:   0%|          | 0/413 [00:00<?, ?it/s]

  Epoch 4/5  loss=0.4469  F1=0.4145
[REL] ✅ Saved → /content/drive/MyDrive/EPOP_documents/models/relation


Epoch 5/5:   0%|          | 0/413 [00:00<?, ?it/s]

  Epoch 5/5  loss=0.3767  F1=0.4032
[REL] Best F1: 0.4145


## 14. Validate on Dev Set

In [15]:
BAD_ENTITY_WORDS = {
    'global', 'world', 'recently', 'area', 'province',
    'country', 'international',
}


def run_pipeline(docs: list, ner_model: PestNERModel,
                 rel_model: RelationModel,
                 rel_threshold: float = 0.35,
                 kg_threshold:  float = 0.20,
                 max_entities:  int   = 50) -> dict:
    """
    Full inference pipeline: text → NER → entity filter → RE → KG edges.

    Returns dict: doc_id → list of edge dicts.
    """
    submission = {}
    for doc in tqdm(docs, desc='Pipeline'):
        try:
            text = doc['raw_text']
            ents = ner_model.predict(text)

            # Filter low-confidence / noisy entities
            filtered, seen_keys = [], set()
            for e in ents:
                t = e['text'].strip()
                if len(t) < 3: continue
                if e['type'] not in ENTITY_TYPES: continue
                if e.get('score', 0) < 0.45: continue
                if t.isdigit() or t.lower() in BAD_ENTITY_WORDS: continue
                key = (t.lower(), e['type'])
                if key in seen_keys: continue
                seen_keys.add(key)
                e['canonical_id'] = t
                filtered.append(e)

            filtered = filtered[:max_entities]
            if len(filtered) < 2:
                submission[doc['id']] = []
                continue

            rels  = rel_model.predict_document(text, filtered, threshold=rel_threshold)
            edges = build_kg(filtered, rels, kg_threshold)
            submission[doc['id']] = edges

        except Exception as exc:
            print(f'[ERROR] {doc["id"]}: {exc}')
            submission[doc['id']] = []

    total = sum(len(v) for v in submission.values())
    print(f'✅ {len(submission)} docs | {total} total edges | '
          f'avg {total / max(1, len(submission)):.2f}/doc')
    return submission


# Reference KG from dev gold edges
dev_refs = {d['id']: d['kg_edges'] for d in dev_docs}
dev_sub  = run_pipeline(dev_docs, ner_model, rel_model)

print('\n=== Dev Set Evaluation ===')
dev_results = evaluate(dev_sub, dev_refs, verbose=True)

Pipeline:   0%|          | 0/55 [00:00<?, ?it/s]

✅ 55 docs | 125 total edges | avg 2.27/doc

=== Dev Set Evaluation ===
  ✗ 103963                              F1=0.0000  pred=0    ref=13
  ✗ 114259                              F1=0.0000  pred=0    ref=41
  ✗ 116119                              F1=0.0000  pred=0    ref=3
  ✗ 99067                               F1=0.0000  pred=0    ref=14
  ✗ 98142                               F1=0.0000  pred=0    ref=5
  ✓ 117892                              F1=1.0000  pred=0    ref=0
  ✗ 107172                              F1=0.0000  pred=6    ref=11
  ✗ 98783                               F1=0.0000  pred=8    ref=20
  ✗ 105611                              F1=0.0000  pred=0    ref=43
  ✗ 98794                               F1=0.0000  pred=0    ref=19
  ✗ 117299                              F1=0.0000  pred=3    ref=14
  ✗ 102488                              F1=0.0000  pred=0    ref=6
  ✗ 106959                              F1=0.0000  pred=0    ref=1
  ✗ 98392                               F1=0.0000 

## 15. Run on Test Set & Generate Submission

In [ ]:
test_sub = run_pipeline(test_docs, ner_model, rel_model)

sub_path = SUBMISSION_DIR / 'submission.csv'
format_submission(test_sub, sub_path)
print(f'\n📁 Submission saved to: {sub_path}')

## 16. Log Results

In [17]:
def log_results(submission: dict, dev_results: dict, output_path) -> None:
    """Save a JSON run report with submission stats and dev F1."""
    total_docs  = len(submission)
    total_edges = sum(len(v) for v in submission.values())
    empty_docs  = sum(1 for v in submission.values() if not v)
    report = {
        'timestamp':         str(datetime.datetime.now()),
        'dev_macro_f1':      dev_results.get('macro_f1', 'N/A'),
        'total_documents':   total_docs,
        'total_edges':       total_edges,
        'avg_edges_per_doc': round(total_edges / max(1, total_docs), 3),
        'empty_docs':        empty_docs,
        'non_empty_docs':    total_docs - empty_docs,
    }
    print('\n📊 ===== PIPELINE RESULTS =====')
    print(json.dumps(report, indent=4))
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump(report, f, indent=4)
    print(f'\n✅ Report saved → {output_path}')

log_results(test_sub, dev_results, SUBMISSION_DIR / 'run_report.json')


📊 ===== PIPELINE RESULTS =====
{
    "timestamp": "2026-05-04 12:07:35.067018",
    "dev_macro_f1": 0.140327,
    "total_documents": 82,
    "total_edges": 259,
    "avg_edges_per_doc": 3.159,
    "empty_docs": 46,
    "non_empty_docs": 36
}

✅ Report saved → /content/drive/MyDrive/EPOP_documents/submissions/run_report.json
